In [ ]:
from customer_ops.domain.schemas import ToolRequest
from customer_ops.llm.client import MissingProviderCredentialError, configured_provider
from customer_ops.llm.experiments import evaluate_tool_sequences
from customer_ops.llm.schemas import ChatMessage, ToolDefinition

try:
    provider = configured_provider()
except MissingProviderCredentialError as error:
    raise RuntimeError(
        "No live result: configure the API key for CUSTOMER_OPS_LLM_PROVIDER and install its extra."
    ) from error
print("provider:", provider.name, "| model:", provider.model)

order_id_parameters = {
    "type": "object",
    "properties": {"order_id": {"type": "string"}},
    "required": ["order_id"],
    "additionalProperties": False,
}
# The model can only propose: the runtime attaches the ticket and policy citation, and an operator approves.
tools = [
    ToolDefinition(
        name="get_order",
        description="Read one accessible order by its ID.",
        parameters=order_id_parameters,
    ),
    ToolDefinition(
        name="propose_cancellation",
        description=(
            "Create a pending cancellation proposal for one accessible order. It does not cancel the order; "
            "an operator must approve the proposal before anything changes."
        ),
        parameters=order_id_parameters,
    ),
]
# Order IDs are database keys, so they must be copied verbatim, including the "order-" prefix.
system_prompt = (
    "You help a support operator by calling tools. Copy order IDs exactly as written in the request, including "
    "the 'order-' prefix. Never invent, shorten, or reformat an ID."
)


def tool_request(name: str, order_id: str) -> ToolRequest:
    return ToolRequest(tool_name=name, arguments={"order_id": order_id})


# Several sequences can be correct: looking the order up first is as valid as proposing straight away.
scenarios = [
    (
        "status-lookup",
        "Find order-gold-10-day status. Stub context: customer is authenticated.",
        [[tool_request("get_order", "order-gold-10-day")]],
    ),
    (
        "cancellation-request",
        "Please cancel order-processing-123. Stub context: customer is authenticated.",
        [
            [tool_request("propose_cancellation", "order-processing-123")],
            [tool_request("get_order", "order-processing-123")],
            [tool_request("get_order", "order-processing-123"), tool_request("propose_cancellation", "order-processing-123")],
        ],
    ),
]
cases = []
for case_id, request_text, valid_sequences in scenarios:
    response = provider.request_tools(
        [ChatMessage(role="system", content=system_prompt), ChatMessage(role="user", content=request_text)], tools
    )
    print(case_id, "->", response.tool_requests)
    cases.append((case_id, response.tool_requests, valid_sequences))
evaluate_tool_sequences(cases).as_dict()

C:\Users\QTF4584\Private\customerops\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


provider: huggingface | model: Qwen/Qwen2.5-1.5B-Instruct


status-lookup -> [ToolRequest(tool_name='get_order', arguments={'order_id': 'order-gold-10-day'})]


cancellation-request -> [ToolRequest(tool_name='propose_cancellation', arguments={'order_id': 'order-12345'})]


{'metric': 'tool_argument_correctness',
 'correct': 1,
 'total': 2,
 'failures': [{'case_id': 'cancellation-request',
   'expected': [[{'tool_name': 'propose_cancellation',
      'arguments': {'order_id': 'order-processing'}}],
    [{'tool_name': 'get_order',
      'arguments': {'order_id': 'order-processing'}}],
    [{'tool_name': 'get_order', 'arguments': {'order_id': 'order-processing'}},
     {'tool_name': 'propose_cancellation',
      'arguments': {'order_id': 'order-processing'}}]],
   'observed': [{'tool_name': 'propose_cancellation',
     'arguments': {'order_id': 'order-12345'}}]}],
 'rate': 0.5}